In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt

from statstech.pipeline import Pipeline
from statstech.preprocessing import StandardScaler
from statstech.model_selection import bootstrap_ci
from statstech.linear_model import LinearRegression, Ridge, Lasso
from statstech.neighbors import KNeighborsRegressor
from statstech.neural_network import MLPRegressor

from analysis.config import SEED
from analysis.io import load_dataset
from analysis.evaluation import evaluate_regressor
from analysis.report import save_figure, save_table

In [ ]:
d = load_dataset("enem")
X_train, y_train = d["X_train"], d["y_train"]
X_test, y_test = d["X_test"], d["y_test"]
features_names = list(d["features_names"]) 

In [ ]:
cols_continuas = None

In [ ]:
def pipe(model):
    return Pipeline([("scaler", StandardScaler(columns=cols_continuas)),
                     ("model", model)])

In [ ]:
final_models = {
    "Linear": pipe(LinearRegression()),
    "Ridge": pipe(Ridge()), 
    "Lasso": pipe(Lasso()),
    "KNN": pipe(KNeighborsRegressor(n_neighbors=121)),
    "Rede": pipe(MLPRegressor(64, 32))
}

In [ ]:
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()

print("Teste aberto no commit:", commit)

In [ ]:
preds = {"Referência": np.full(len(y_test), y_train.mean())}

for name, model in final_models.items():
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)

lines = []
for name, pred in preds.items():
    lines.append({"model": name, **evaluate_regressor(y_test, pred)})

for l in lines:
    print(l)
    


In [ ]:
def rmse(yt, yp):
    return evaluate_regressor(yt, yp)["rmse"]

table = []
for name, pred in preds.items():
    est, low, high = bootstrap_ci((y_test, pred), rmse,
                                   n_bootstrap=1000, level=0.95, random_state=SEED)
    met = evaluate_regressor(y_test, pred)
    table.append({"model":name,
                  "rmse": est, 
                  "rmse_low": low, 
                  "rmse_high": high,
                  "mae": met["mae"],
                  "r2": met["r2"], 
                  "commit_test": commit})

save_table(table, "enem_teste_final")

In [ ]:
est, low, high = bootstrap_ci((y_test, preds["Rede"], preds["Ridge"]), 
                              lambda yt, a, b: rmse(yt, a) - rmse(yt, b),
                                   n_bootstrap=1000, level=0.95, random_state=SEED)

print(f"RMSE(rede) - RMSE(Ridge) = {est: .2f} IC95% [{low: .2f}, {high: .2f}]")
save_table([{"comparacao": "Rede - Ridge", "diferenca_rmse": est,
            "low": low, "high": high}],"enem_rede_vs_ridge")

In [ ]:
names = [t["model"] for t in table]

r = np.array([t["rmse"] for t in table])

err = np.array([[t["rmse"] - t["rmse_low"] for t in table],
                [t["rmse_high"] - t["rmse"] for t in table]])

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(names, r, yerr= err, capsize=4)
ax.set_title("RMSE no teste, com IC de 95% (bootstrap)")
ax.set_ylabel("RMSE (pontos)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_figure(fig, "enem_comparacao")
plt.show()

In [ ]:
best = min(table[1:], key=lambda t: t["rmse"])["model"]
pred_best = preds[best]

print("Melhor modelo de teste:", best)

In [ ]:
school_idx = features_names.index("school_3")
region_idx = features_names.index("region_Sul")
income_idx = features_names.index("log_income_per_capita")


q1, q3 = np.quantile(X_train[:, income_idx], [0.25, 0.75])

groups = {
    "Escola privada": X_test[:, school_idx] == 1,
    "Escola pública": X_test[:, school_idx] == 0,

    "Renda (Q1)": X_test[:, income_idx] <= q1,
    "Renda (Q4)": X_test[:, income_idx] >= q3,

    "Região Sul": X_test[:, region_idx] == 1,
    "Demais regiões": X_test[:, region_idx] == 0,
}


rows_group = []
for g, mask in groups.items():
    rows_group.append({"grupo": g, "n": int(mask.sum()),
                       "rmse": rmse(y_test[mask], pred_best[mask])})
 
if rows_group:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar([r_["grupo"] for r_ in rows_group], [r_["rmse"] for r_ in rows_group])
    ax.set_title(f"RMSE por grupo ({best})")
    ax.set_ylabel("RMSE (pontos)")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    save_figure(fig, "enem_rmse_grupo")
    save_table(rows_group, "enem_rmse_grupo")
    plt.show()

In [ ]:
residual = y_test - pred_best

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(pred_best, residual, s=6, alpha=0.3)
ax.axhline(0, linewidth=1)
ax.set_title(f"Resíduos x previsto ({best})")
ax.set_xlabel("Nota prevista (pontos)")
ax.set_ylabel("Resíduo: observado - previsto (pontos)")
ax.grid(alpha=0.3)
plt.tight_layout()
save_figure(fig, "enem_residuos")
plt.show()

In [ ]:
lasso_model = final_models["Lasso"].named_steps_["model"]
coef = lasso_model.coef_

maintained = [{"variavel": n, "coeficiente": float(c)}
              for n, c in zip(features_names, coef) if c != 0]
maintained = sorted(maintained, key=lambda r_: -abs(r_["coeficiente"]))

print(f"O Lasso manteve {len(maintained)} de {len(features_names)} variáveis")
save_table(maintained, "enem_lasso_mantidas")